# Membryo preprocessing

This public notebook was migrated from the audited read-only research source. Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'Membryo'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import scanpy as sc
import scvi, os

In [ ]:
adata = sc.read_h5ad(str(DATA_ROOT / 'Mouse_embryo_all_stage.h5ad'))
adata

In [ ]:
adata.obs["timepoint"].value_counts().sort_index()

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000, flavor="seurat_v3")

sc.tl.pca(adata, n_comps=50)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color="annotation", wspace=0.4)

In [ ]:
sc.pl.umap(adata, color="timepoint", wspace=0.4)

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

adata.obs["annotation"] = adata.obs["annotation"].astype("category")

coords = np.asarray(adata.obsm["spatial"])
x = coords[:, 0]
y = coords[:, 1]

ann_cats = list(adata.obs["annotation"].cat.categories)
if "annotation_colors" in adata.uns and len(adata.uns["annotation_colors"]) >= len(ann_cats):
    ann_colors = list(adata.uns["annotation_colors"][:len(ann_cats)])
else:
    cmap = plt.get_cmap("tab20")
    ann_colors = [cmap(i % 20) for i in range(len(ann_cats))]

color_map = dict(zip(ann_cats, ann_colors))
labels = adata.obs["annotation"].astype(str).values
c = [color_map[a] for a in labels]

fig, ax = plt.subplots(figsize=(10, 10), dpi=180)
ax.scatter(x, y, c=c, s=0.4, alpha=0.7, linewidths=0, rasterized=True)

ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()   
ax.set_frame_on(False) 

handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=color_map[k], markersize=5, label=k)
    for k in ann_cats
]
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=min(8, len(ann_cats)),
    frameon=False,
    fontsize=8
)

plt.tight_layout(rect=[0, -0.4, 1, 1])
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================
# 0) 
# =========================
USE_ALIGNED_COORDS = False   # True= cx_aligned/cy_aligned；False= obsm['spatial']
OUT_COL = "is_brain_microenv_by_brainbottom"

# “”
BRAIN_MICROENV_ANN = [
    "Brain",
    "Choroid plexus",
    "Meninges",
    "Head mesenchyme",
    "Neural crest",
    "Blood vessel",
    "Cavity",
    # "Connective tissue",  # 
]

#  quantile（ timepoint ）
BRAIN_BOTTOM_Q_DEFAULT = 0.995

#  timepoint 
BRAIN_BOTTOM_Q_BY_TP = {
    "E10.5": 0.95,
}

# （=）
# >0: （Brain）
# <0: 
Y_MARGIN = 0.0

# （）， False 
# True  :  y <= （ y ）
# False :  y >= （ y ）
KEEP_ABOVE_BRAIN_BOTTOM = True


# =========================
# 1) 
# =========================
obs = adata.obs.copy()

if USE_ALIGNED_COORDS and {"cx_aligned", "cy_aligned"}.issubset(obs.columns):
    XY = obs[["cx_aligned", "cy_aligned"]].to_numpy(dtype=np.float32)
    print("[coords] using aligned coords")
else:
    XY = np.asarray(adata.obsm["spatial"][:, :2], dtype=np.float32)
    print("[coords] using raw spatial coords")

x_all = XY[:, 0]
y_all = XY[:, 1]

tp_all = obs["timepoint"].astype(str).to_numpy()
ann_all = obs["annotation"].astype(str).to_numpy()

# timepoint （）
def _tp_sort_key(s):
    import re
    m = re.search(r"(\d+(?:\.\d+)?)", str(s))
    return float(m.group(1)) if m else 1e9

timepoints = sorted(pd.unique(tp_all), key=_tp_sort_key)
print("[timepoints]", timepoints)


# =========================
# 2)  timepoint  Brain 
#     y “”（MOSTA/）
# =========================
is_keep = np.zeros(adata.n_obs, dtype=bool)
bottom_by_tp = {}

for tp in timepoints:
    m_tp = (tp_all == tp)
    m_brain = m_tp & (ann_all == "Brain")

    if m_brain.sum() == 0:
        print(f"[skip] {tp}: no Brain cells")
        continue

    y_brain = y_all[m_brain]

    #  timepoint  quantile（E10.5=0.95，=0.99）
    q_tp = float(BRAIN_BOTTOM_Q_BY_TP.get(str(tp), BRAIN_BOTTOM_Q_DEFAULT))

    # Brain（bottom）
    if q_tp >= 1.0:
        y_bottom = float(np.max(y_brain))
    else:
        y_bottom = float(np.quantile(y_brain, q_tp))

    y_thr = y_bottom + float(Y_MARGIN)
    bottom_by_tp[tp] = y_thr

    # 
    m_ann = np.isin(ann_all, BRAIN_MICROENV_ANN)

    # “Brain”（ y <= y_thr）
    if KEEP_ABOVE_BRAIN_BOTTOM:
        m_keep_tp = m_tp & m_ann & (y_all <= y_thr)
    else:
        m_keep_tp = m_tp & m_ann & (y_all >= y_thr)

    is_keep |= m_keep_tp

    print(
        f"[{tp}] q={q_tp:.3f} | Brain n={m_brain.sum():>6d} | "
        f"y_bottom={y_bottom:.2f} | thr={y_thr:.2f} | selected={m_keep_tp.sum():>6d}"
    )

#  obs（ layers）
adata.obs[OUT_COL] = is_keep

# 
adata_brain_microenv = adata[is_keep].copy()

print("\n=== Done ===")
print(adata_brain_microenv)
print("\nCounts by timepoint:")
print(adata_brain_microenv.obs["timepoint"].value_counts().sort_index())
print("\nCounts by annotation:")
print(adata_brain_microenv.obs["annotation"].value_counts())

# =========================
# 3) （ annotation ）
# =========================
ann_sel = ann_all[is_keep]
XY_sel = XY[is_keep]

# （ annotation_colors）
all_ann_cats = list(pd.Categorical(obs["annotation"].astype(str)).categories)

if "annotation_colors" in adata.uns and len(adata.uns["annotation_colors"]) >= len(all_ann_cats):
    full_map = dict(zip(all_ann_cats, adata.uns["annotation_colors"][:len(all_ann_cats)]))
    cats_sel = list(pd.Categorical(ann_sel).categories)
    color_map = {c: full_map.get(c, "gray") for c in cats_sel}
else:
    cmap = plt.get_cmap("tab20")
    cats_sel = list(pd.Categorical(ann_sel).categories)
    color_map = {c: cmap(i % 20) for i, c in enumerate(cats_sel)}

colors_sel = [color_map[a] for a in ann_sel]

fig, ax = plt.subplots(figsize=(12, 5), dpi=180)
ax.scatter(
    XY_sel[:, 0], XY_sel[:, 1],
    c=colors_sel, s=0.35, alpha=0.85, linewidths=0, rasterized=True
)

for tp in timepoints:
    if tp not in bottom_by_tp:
        continue
    m_tp = (tp_all == tp)
    x_tp = x_all[m_tp]
    if x_tp.size == 0:
        continue
    y_thr = bottom_by_tp[tp]
    # ax.hlines(y=y_thr, xmin=x_tp.min(), xmax=x_tp.max(), linewidth=1)

# ax.set_title(f"Brain microenvironment by Brain-bottom threshold (N={XY_sel.shape[0]})")
ax.set_aspect("equal")
ax.invert_yaxis()
ax.axis("off")

handles = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=color_map[c], markersize=5, label=c)
    for c in cats_sel
]
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=min(8, len(handles)),
    frameon=False,
    fontsize=8
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
scvi.model.SCANVI.setup_anndata(
    adata_brain_microenv,
    layer="count",
    labels_key="annotation",
    unlabeled_category="Unknown",   
    batch_key="timepoint"
)
model = scvi.model.SCANVI(adata_brain_microenv)
model.train()
model_dir = SCANVI_DIR


In [ ]:
model_dir = SCANVI_DIR


In [ ]:
model


In [ ]:
SCVI_LATENT_KEY = "X_scanVI"

latent = model.get_latent_representation()
adata_brain_microenv.obsm[SCVI_LATENT_KEY] = latent
latent.shape

In [ ]:
sc.pp.neighbors(adata_brain_microenv, use_rep=SCVI_LATENT_KEY)
sc.tl.umap(adata_brain_microenv)

In [ ]:
sc.pl.umap(adata_brain_microenv, color=["annotation"],frameon=False,size = 5,)

In [ ]:
sc.pl.umap(adata_brain_microenv,color=["timepoint"],frameon=False,size = 5,)

In [ ]:
import re

TIMEPOINT_COL = "timepoint"
ANN_COL = "annotation"
SPATIAL_KEY = "spatial"

REF_TP = "E16.5"

# “”：
# - "mean"   = （）
# - "median" = （）
CENTER_METHOD = "mean"

#  Brain （ True，）
USE_ONLY_BRAIN_FOR_CENTER = True
BRAIN_LABEL = "Brain"

# =========================
# 2) 
# =========================
if SPATIAL_KEY not in adata_brain_microenv.obsm:
    raise KeyError(f"adata_brain_microenv.obsm['{SPATIAL_KEY}'] ")

if TIMEPOINT_COL not in adata_brain_microenv.obs.columns:
    raise KeyError(f"adata_brain_microenv.obs['{TIMEPOINT_COL}'] ")

XY = np.asarray(adata_brain_microenv.obsm[SPATIAL_KEY][:, :2], dtype=np.float32)  # [x, y]
tp_all = adata_brain_microenv.obs[TIMEPOINT_COL].astype(str).to_numpy()

if ANN_COL in adata_brain_microenv.obs.columns:
    ann_all = adata_brain_microenv.obs[ANN_COL].astype(str).to_numpy()
else:
    ann_all = np.array([""] * adata_brain_microenv.n_obs, dtype=object)


def _tp_sort_key(s):
    m = re.search(r"(\d+(?:\.\d+)?)", str(s))
    return float(m.group(1)) if m else 1e9

timepoints = sorted(pd.unique(tp_all), key=_tp_sort_key)
print("Timepoints:", timepoints)

if str(REF_TP) not in set(timepoints):
    raise ValueError(f" {REF_TP}  timepoint ， REF_TP")

# =========================
# 3)  timepoint 
# =========================
def _center_of_points(P, method="mean"):
    if P.shape[0] == 0:
        return None
    if method == "mean":
        return P.mean(axis=0).astype(np.float32)
    elif method == "median":
        return np.median(P, axis=0).astype(np.float32)
    else:
        raise ValueError("CENTER_METHOD  'mean'  'median'")

center_by_tp = {}

for tp in timepoints:
    m_tp = (tp_all == str(tp))

    #  timepoint 
    P_tp = XY[m_tp]

    # ： Brain 
    if USE_ONLY_BRAIN_FOR_CENTER and ANN_COL in adata_brain_microenv.obs.columns:
        m_brain = m_tp & (ann_all == BRAIN_LABEL)
        if m_brain.sum() > 0:
            P_center = XY[m_brain]
            used = f"{BRAIN_LABEL} only"
        else:
            P_center = P_tp
            used = "all cells (fallback, no Brain)"
    else:
        P_center = P_tp
        used = "all cells"

    c = _center_of_points(P_center, method=CENTER_METHOD)
    center_by_tp[str(tp)] = c

    print(f"[{tp}] n={P_tp.shape[0]:6d} | center=({c[0]:.2f}, {c[1]:.2f}) | used={used}")

c_ref = center_by_tp[str(REF_TP)]
print(f"\nReference: {REF_TP}, center = ({c_ref[0]:.2f}, {c_ref[1]:.2f})")

XY_aligned = XY.copy()
shift_by_tp = {}

for tp in timepoints:
    m_tp = (tp_all == str(tp))
    c_tp = center_by_tp[str(tp)]
    shift = (c_ref - c_tp).astype(np.float32)  # [dx, dy]
    shift_by_tp[str(tp)] = shift

    XY_aligned[m_tp, 0] = XY[m_tp, 0] + shift[0]
    XY_aligned[m_tp, 1] = XY[m_tp, 1] + shift[1]

    print(f"[{tp}] shift dx={shift[0]: .2f}, dy={shift[1]: .2f}")

adata_brain_microenv.obs["cx_aligned"] = XY_aligned[:, 0].astype(np.float32)
adata_brain_microenv.obs["cy_aligned"] = XY_aligned[:, 1].astype(np.float32)

print("\nDone.")
print("Written to:")
print("  adata_brain_microenv.obs['cx_aligned']")
print("  adata_brain_microenv.obs['cy_aligned']")

adata_brain_microenv.obsm["spatial_aligned_centroid"] = XY_aligned.astype(np.float32)
print("Optional saved to:")
print("  adata_brain_microenv.obsm['spatial_aligned_centroid']")


fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
ax1, ax2 = axes

tp_cats = list(timepoints)
cmap = plt.get_cmap("tab20")
tp_color = {tp: cmap(i % 20) for i, tp in enumerate(tp_cats)}
colors = [tp_color[t] for t in tp_all]

ax1.scatter(XY[:, 0], XY[:, 1], c=colors, s=0.3, alpha=0.6, linewidths=0, rasterized=True)
ax1.set_title("Before centroid alignment")
ax1.set_aspect("equal")
ax1.invert_yaxis()
ax1.axis("off")

ax2.scatter(XY_aligned[:, 0], XY_aligned[:, 1], c=colors, s=0.8, alpha=0.6, linewidths=0, rasterized=True)
ax2.set_title(f"After centroid alignment (ref={REF_TP})")
ax2.set_aspect("equal")
ax2.invert_yaxis()
ax2.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model.save(model_dir, overwrite=True, save_anndata=True)
print(f"Saved model-ready AnnData: {SCANVI_ADATA}")
